# 01 — Exploring ALeRCE Alerts

This notebook walks through the ALeRCE data model and helps you build
intuition for what astronomical alerts look like before writing any
pipeline code.

**No API key needed. No database needed. Just run the cells.**

In [ ]:
# Setup
from alerce.core import Alerce
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

client = Alerce()
print('Connected to ALeRCE')

## 1. What classes of objects does ALeRCE track?

ALeRCE uses two classifiers:
- **Stamp classifier**: fast, uses the image cutout ("stamp") around each alert
- **Light curve classifier**: more accurate, uses the full brightness-over-time history

The main classes relevant to us (transients, not variables):
- **SNIa** — Type Ia supernovae (exploding white dwarfs, used to measure the universe's expansion)
- **SNII** — Type II supernovae (massive star core collapse)
- **SNIbc** — Type Ib/c supernovae (stripped-envelope core collapse)
- **SLSN** — Superluminous supernovae (extremely bright, rare)
- **TDE** — Tidal disruption events (a star shredded by a black hole)
- **AGN** — Active galactic nuclei (supermassive black hole accretion)
- **KN** — Kilonovae (neutron star mergers — gravitational wave counterparts!)

In [ ]:
# Query recent Type Ia supernovae (the most common and well-understood type)
sne = client.query_objects(
    classifier='lc_classifier',
    class_name='SNIa',
    format='pandas',
    page_size=20,
    probability=0.8,  # Only high-confidence classifications
)

print(f'Got {len(sne)} Type Ia supernova candidates')
print(f'\nColumns available: {list(sne.columns)}')
sne[['oid', 'meanra', 'meandec', 'firstmjd', 'lastmjd', 'ndet']].head(10)

## 2. What does a light curve look like?

A **light curve** is brightness vs. time. For a supernova, you'll see:
- A sudden brightening (the explosion)
- A peak
- A gradual decline

Astronomers measure brightness in **magnitudes** (confusingly, *lower = brighter*).
The data comes in multiple **bands** (color filters): g-band (green/blue) and r-band (red) for ZTF.

In [ ]:
# Pick the first supernova and plot its light curve
oid = sne.iloc[0]['oid']
detections = client.query_detections(oid, format='pandas', sort='mjd')

print(f'Object: {oid}')
print(f'Detections: {len(detections)}')
print(f'Time span: MJD {detections["mjd"].min():.1f} to {detections["mjd"].max():.1f}')

fig, ax = plt.subplots()

colors = {1: '#4fc3f7', 2: '#ef5350'}  # g=blue, r=red
labels = {1: 'g-band', 2: 'r-band'}

for fid, group in detections.groupby('fid'):
    if fid not in colors:
        continue
    ax.errorbar(
        group['mjd'], group['magpsf'],
        yerr=group['sigmapsf'],
        fmt='o', color=colors[fid], label=labels[fid],
        markersize=5, alpha=0.8, capsize=2
    )

ax.invert_yaxis()  # Brighter = lower magnitude (astronomers...)
ax.set_xlabel('Modified Julian Date (MJD)')
ax.set_ylabel('Magnitude (lower = brighter)')
ax.set_title(f'Light Curve: {oid} (Type Ia Supernova Candidate)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Where are these objects on the sky?

Sky positions use **Right Ascension (RA)** and **Declination (Dec)** — think longitude and latitude but for the sky. RA goes 0-360 degrees, Dec goes -90 to +90.

In [ ]:
# Plot supernova positions on the sky (Mollweide projection)
fig, ax = plt.subplots(subplot_kw={'projection': 'mollweide'})

# Convert RA from degrees to radians, centered on 0
ra_rad = np.radians(sne['meanra'].values)
ra_rad = np.where(ra_rad > np.pi, ra_rad - 2*np.pi, ra_rad)
dec_rad = np.radians(sne['meandec'].values)

ax.scatter(ra_rad, dec_rad, c='#ff7043', s=50, alpha=0.8, edgecolors='white', linewidths=0.5)
ax.grid(True, alpha=0.3)
ax.set_title(f'{len(sne)} Recent Type Ia Supernova Candidates', pad=20)
plt.tight_layout()
plt.show()

## 4. Cross-matching with SIMBAD

For each alert, we want to know: is this a *known* object? Is it near a galaxy?
SIMBAD is a database of millions of known astronomical objects.

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.simbad import Simbad

# Take our first supernova and check SIMBAD
test_obj = sne.iloc[0]
coord = SkyCoord(ra=test_obj['meanra'], dec=test_obj['meandec'], unit=(u.degree, u.degree))

print(f'Searching SIMBAD around {test_obj["oid"]}...')
print(f'Position: RA={test_obj["meanra"]:.4f}, Dec={test_obj["meandec"]:.4f}')

result = Simbad.query_region(coord, radius=30 * u.arcsec)

if result:
    print(f'\nFound {len(result)} SIMBAD matches within 30 arcseconds:')
    for row in result:
        print(f'  {row["MAIN_ID"]:30s}  type: {row["OTYPE"]}')
else:
    print('\nNo SIMBAD match — this could be a genuinely new transient!')

## 5. Comparing different transient types

Let's pull a few different classes and see how their light curves differ.

In [ ]:
classes_to_compare = ['SNIa', 'SNII', 'AGN']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, cls in zip(axes, classes_to_compare):
    try:
        objs = client.query_objects(
            classifier='lc_classifier', class_name=cls,
            format='pandas', page_size=3, probability=0.8
        )
        if objs is not None and len(objs) > 0:
            test_oid = objs.iloc[0]['oid']
            dets = client.query_detections(test_oid, format='pandas', sort='mjd')
            
            for fid, group in dets.groupby('fid'):
                color = '#4fc3f7' if fid == 1 else '#ef5350'
                label = 'g' if fid == 1 else 'r'
                ax.errorbar(group['mjd'], group['magpsf'], yerr=group['sigmapsf'],
                           fmt='o', color=color, label=label, markersize=4, alpha=0.7, capsize=2)
            
            ax.invert_yaxis()
            ax.set_title(f'{cls}: {test_oid}', fontsize=11)
            ax.set_xlabel('MJD')
            ax.legend(fontsize=9)
    except Exception as e:
        ax.text(0.5, 0.5, f'{cls}\n(no data)', ha='center', va='center', transform=ax.transAxes)

axes[0].set_ylabel('Magnitude')
fig.suptitle('Light Curve Comparison: Different Transient Types', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\nNotice the differences:')
print('  SNIa — Sharp rise, smooth decline (standard candle for cosmology)')
print('  SNII — Plateau phase after peak (hydrogen envelope)')
print('  AGN  — Irregular flickering (accretion disk around supermassive black hole)')

## Next Steps

You now understand the core data model:
- **Objects** have sky positions (RA/Dec), classifications, and timestamps
- **Detections** are individual brightness measurements forming a light curve
- **Cross-matches** connect alerts to known objects in catalogs like SIMBAD

This is exactly what the Rubin Scout ingestion pipeline does automatically.

Move on to notebook `02_light_curve_analysis.ipynb` for feature extraction,
or start the backend with `docker-compose up` and explore the API at `/docs`.